In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


In [3]:
!pip install kagglehub


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\YuTech\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gzxjrisal/semi-supervised-banking-transaction-dataset")

print("Path to dataset files:", path)

C:\Users\YuTech\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\YuTech\.cache\kagglehub\datasets\gzxjrisal\semi-supervised-banking-transaction-dataset\versions\1


In [5]:
import os

for f in os.listdir(path):
    print(f)

transactions_kaggle.csv


In [6]:
df = pd.read_csv(os.path.join(path, "transactions_kaggle.csv"))
print(df.shape)
df.head()

(33868, 7)


,transaction_id,persona_id,date,description,amount,category,is_labelled
0,1,P01,01/09/2023,Balance brought forward,0.00,Excluded,True
1,2,P01,01/09/2023,DIRECT DEBIT BUPA AUSTRALIA DDR ID 098647,120.90,Essential,True
2,3,P01,01/09/2023,VISA PURCHASE HENRY COFFEE FORTITUDE VLY,6.19,Non-Essential,True
3,4,P01,02/09/2023,VISA PURCHASE SUSHI HUB CBD,9.11,Non-Essential,True
4,5,P01,02/09/2023,POS CAMPOS SOUTH BRISBANE TID 957282,6.29,Non-Essential,True


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
df_model = df[df['category'].isin(['Essential', 'Non-Essential'])].copy()

X = df_model['description']
y = df_model['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

               precision    recall  f1-score   support

    Essential       1.00      0.95      0.97       141
Non-Essential       0.99      1.00      1.00       832

     accuracy                           0.99       973
    macro avg       1.00      0.98      0.99       973
 weighted avg       0.99      0.99      0.99       973



In [10]:
def predict_importance(description: str) -> str:
    vec = vectorizer.transform([description])
    pred = model.predict(vec)[0]
    return "muhim" if pred == "Essential" else "muhimmas"

print(predict_importance("VISA PURCHASE HENRY COFFEE FORTITUDE VLY"))
print(predict_importance("DIRECT DEBIT BUPA AUSTRALIA DDR ID 098647"))

muhimmas
muhim


In [11]:
def predict_importance(description: str) -> str:
    vec = vectorizer.transform([description])
    pred = model.predict(vec)[0]
    return "muhim" if pred == "Essential" else "muhimmas"

tranzaksiyalar = [
    "VISA PURCHASE HENRY COFFEE FORTITUDE VLY",
    "DIRECT DEBIT BUPA AUSTRALIA DDR ID 098647"
]

for t in tranzaksiyalar:
    natija = predict_importance(t)
    print(f"Tranzaksiya: {t}")
    print(f"Natija: Bu xarajat — {natija}")
    print("-" * 50)

Tranzaksiya: VISA PURCHASE HENRY COFFEE FORTITUDE VLY
Natija: Bu xarajat — muhimmas
--------------------------------------------------
Tranzaksiya: DIRECT DEBIT BUPA AUSTRALIA DDR ID 098647
Natija: Bu xarajat — muhim
--------------------------------------------------


In [12]:
matn = input("Tranzaksiya tavsifini kiriting: ")
natija = predict_importance(matn)
print(f"Bu xarajat — {natija}")

Tranzaksiya tavsifini kiriting:  VISA PURCHASE SUSHI HUB CBD


Bu xarajat — muhimmas
